# LangGraph 핸즈온 — 고객지원 이메일 에이전트

> **AI 오딧세이 세미나 · "노드·엣지·State 편"** 의 실습 노트북.
> 리포트의 5단계 설계를 **위에서 아래로 실행**하며 하나의 이메일 에이전트를 완성합니다.

이 노트북은 리포트 본문(서사)과 `scripts/` 의 코드(실행)를 한곳에 합친 것입니다. 각 셀은 리포트의 한 절에 대응합니다.

| 단계 | 리포트 | 이 노트북에서 |
|---|---|---|
| ① 분해 | §2 | 업무를 노드로 쪼개고 최소 그래프를 돌린다 |
| ② 스텝 유형 | §2.2 | 네 가지 작업 유형으로 분류 (LLM·데이터·액션·사용자입력) |
| ③ State 설계 | §3.1-3.2 | `EmailAgentState` — raw 데이터만, 포맷은 노드에서 |
| ④ 노드 구현 + 에러 | §3.3-3.4 | 7개 노드 + 에러 4전략 |
| ⑤ 연결 + 사람 개입 | §4 | `Command(goto)` 라우팅 · `interrupt`/resume · checkpointer |

> **API 키 없이 돌아갑니다.** `OPENAI_API_KEY` 가 있으면 실제 LLM(`gpt-5-nano`), 없으면 규칙 기반 시뮬레이션으로 동작합니다.

## 0. 준비 — 패키지와 LLM 옵셔널 헬퍼

먼저 의존성을 확인합니다. 이미 설치돼 있으면 아래 설치 셀은 건너뛰어도 됩니다. 그다음, 분류·초안 생성을 **키가 있으면 실제 LLM, 없으면 규칙 기반 시뮬레이션**으로 처리하는 헬퍼를 정의합니다. 이 패턴 덕분에 세미나에서 키 없이도 모든 셀이 실행됩니다.

In [ ]:
# 필요시 설치 (이미 깔려 있으면 satisfied 로 건너뜀)
%pip install -q "langgraph>=1.0" langchain-openai python-dotenv

In [ ]:
import os
from typing import Literal, TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command, RetryPolicy, interrupt

# .env 가 있으면 로드 (없어도 무방)
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass


def get_llm():
    """OPENAI_API_KEY 가 있고 패키지가 깔려 있으면 ChatOpenAI, 아니면 None."""
    if not os.getenv("OPENAI_API_KEY"):
        return None
    try:
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(model="gpt-5-nano")  # 리포트가 쓴 모델
    except ImportError:
        return None


_mode = "실제 LLM (ChatOpenAI)" if get_llm() is not None else "규칙 기반 시뮬레이션 (OPENAI_API_KEY 없음)"
print(f"분류 모드: {_mode}")

## 1. 분해 + 스텝 유형 — 업무를 노드로 쪼갠다 (§2)

LangGraph 설계의 첫걸음은 업무를 **개별 스텝(노드)** 으로 나누는 것입니다. 노드 하나는 "한 가지 일만 하는 함수"이고, 각 노드는 네 가지 **스텝 유형** 중 하나입니다 — 🧠 LLM(이해·생성), 🗄️ 데이터(외부 조회), ⚡ 액션(외부 행동), 👤 사용자 입력(사람 개입). 이메일 에이전트는 7개 노드로 분해됩니다: `read_email → classify_intent → {search_documentation · bug_tracking} → draft_response → human_review → send_reply`.

## 2. State 설계 — raw 데이터만 담는다 (§3.1-3.2)

State 는 모든 노드가 공유하는 **메모리**입니다. 담을지 말지는 한 가지 질문으로 정합니다 — *"단계를 넘어 보존돼야 하는가?"* 핵심 원칙은 **포맷된 문자열이 아니라 raw 데이터를 저장**하는 것입니다. 그래야 노드마다 같은 데이터를 다르게 포맷해 쓰고, 프롬프트를 바꿔도 스키마가 흔들리지 않습니다. `total=False` 로 둬서 초기 State 에 모든 키를 채우지 않아도 되게 합니다(리포트 §3.4 의 주의).

In [ ]:
class EmailClassification(TypedDict):
    """분류 결과 — LLM 이 준 그대로 '하나의 딕셔너리' 로 저장한다."""
    intent: Literal["question", "bug", "billing", "feature", "complex"]
    urgency: Literal["low", "medium", "high", "critical"]
    topic: str
    summary: str


class EmailAgentState(TypedDict, total=False):
    # 원본 (나중에 재구성 불가 → 담는다)
    email_content: str
    sender_email: str
    email_id: str
    # 분류 결과 (이후 여러 노드가 사용)
    classification: EmailClassification | None
    # 원본 검색/조회 결과 (다시 가져오기 비쌈)
    search_results: list[str] | None
    # 생성된 콘텐츠
    draft_response: str | None
    # 실행 로그 (관찰·디버깅용)
    log: list[str]


def log(state, line):
    """누적 로그 헬퍼 — log 에 한 줄 덧붙인 새 리스트를 돌려준다."""
    return [*state.get("log", []), line]

print("State 스키마 정의 완료:", list(EmailAgentState.__annotations__))

## 3. 노드 구현 + 에러 4전략 — 라우팅은 노드 안에서 (§3.3-3.4)

노드는 **State 를 받아 업데이트를 돌려주는 함수**입니다. 분기가 필요한 노드는 `Command(goto=...)` 로 **다음 행선지까지 스스로 결정**하고, `Command[Literal[...]]` 타입 힌트로 갈 수 있는 곳을 선언합니다. 에러는 종류마다 다르게 다룹니다 — 아래 코드에 네 전략이 들어 있습니다: ① `search_documentation` 에 거는 `RetryPolicy`(재시도), ③ `human_review` 의 `interrupt()`(일시정지), ④ `send_reply` 의 `raise`(띄워보내기). ②(LLM 복구·되돌아오기)는 도구 루프 패턴이라 §3.3 끝의 별도 데모에서 봅니다.

In [ ]:
def _simulate_classification(email_content: str) -> EmailClassification:
    """LLM 이 없을 때 쓰는 규칙 기반 분류 — 키 없이도 돌게 한다."""
    text = email_content.lower()
    if any(k in email_content for k in ("청구", "환불", "결제", "요금")) or "billing" in text:
        return {"intent": "billing", "urgency": "critical", "topic": "청구/결제", "summary": "청구 관련 긴급 문의"}
    if any(k in email_content for k in ("버그", "오류", "안 돼", "안돼", "에러")) or "bug" in text:
        return {"intent": "bug", "urgency": "high", "topic": "버그 리포트", "summary": "제품 오류 신고"}
    if any(k in email_content for k in ("어떻게", "방법", "?", "？", "문의")):
        return {"intent": "question", "urgency": "low", "topic": "사용 문의", "summary": "기능 사용법 질문"}
    return {"intent": "complex", "urgency": "medium", "topic": "기타", "summary": "분류 모호"}


def read_email(state: EmailAgentState) -> dict:
    """[데이터 스텝] 이메일 수신. 늘 classify_intent 로 간다(고정 엣지)."""
    return {"log": log(state, f"📥 read_email: {state['email_id']} 수신")}


def classify_intent(
    state: EmailAgentState,
) -> Command[Literal["search_documentation", "human_review", "bug_tracking", "draft_response"]]:
    """[LLM 스텝] 의도·긴급도를 분류하고 그에 맞게 라우팅."""
    llm = get_llm()
    if llm is not None:
        prompt = (f"이 고객 이메일을 분석해 분류하라:\n\nEmail: {state['email_content']}\n"
                  f"From: {state.get('sender_email', '')}\n\n"
                  "intent(question/bug/billing/feature/complex), urgency(low/medium/high/critical), "
                  "topic, summary 를 포함해 분류하라.")
        classification = llm.with_structured_output(EmailClassification).invoke(prompt)
    else:
        classification = _simulate_classification(state["email_content"])

    if classification["intent"] == "billing" or classification["urgency"] == "critical":
        goto = "human_review"
    elif classification["intent"] in ("question", "feature"):
        goto = "search_documentation"
    elif classification["intent"] == "bug":
        goto = "bug_tracking"
    else:
        goto = "draft_response"

    line = f"🧠 classify_intent: intent={classification['intent']} urgency={classification['urgency']} → {goto}"
    return Command(update={"classification": classification, "log": log(state, line)}, goto=goto)


def search_documentation(state: EmailAgentState) -> Command[Literal["draft_response"]]:
    """[데이터 스텝] 지식 베이스 검색. 일시적 실패가 잦아 RetryPolicy 를 건다(전략 ①)."""
    topic = (state.get("classification") or {}).get("topic", "일반")
    results = [f"[문서] '{topic}' 도움말 #1", f"[문서] '{topic}' FAQ #2"]
    return Command(update={"search_results": results, "log": log(state, f"🗄️ search_documentation: {len(results)}건")},
                   goto="draft_response")


def bug_tracking(state: EmailAgentState) -> Command[Literal["draft_response"]]:
    """[액션 스텝] 이슈 생성. 캐시하지 않는다 — 매 호출이 고유 행동."""
    issue_id = f"BUG-{state['email_id'][-3:]}"
    return Command(update={"search_results": [f"이슈 {issue_id} 생성됨"], "log": log(state, f"⚡ bug_tracking: {issue_id}")},
                   goto="draft_response")


def draft_response(state: EmailAgentState) -> Command[Literal["human_review"]]:
    """[LLM 스텝] 답변 초안 생성. 작성 후 항상 human_review 로."""
    cls = state.get("classification") or {}
    context = "\n".join(state.get("search_results") or [])
    llm = get_llm()
    if llm is not None:
        prompt = (f"고객 이메일에 대한 정중한 한국어 답변 초안을 작성하라.\n\n"
                  f"원문: {state['email_content']}\n분류: {cls}\n참고 자료:\n{context}")
        draft = llm.invoke(prompt).content
    else:
        draft = f"안녕하세요, 문의 주신 '{cls.get('topic', '내용')}' 건 확인했습니다. 아래 자료를 참고해 처리해 드리겠습니다.\n{context}".strip()
    return Command(update={"draft_response": draft, "log": log(state, "✍️ draft_response: 초안 작성")}, goto="human_review")


def human_review(state: EmailAgentState) -> Command[Literal["send_reply", END]]:
    """[사용자 입력 스텝] interrupt 로 멈추고 사람의 결정을 받는다(전략 ③).

    interrupt() 이전에는 부작용 있는 호출을 두지 않는다 — 재개 시 이 앞은 다시 실행된다.
    """
    cls = state.get("classification") or {}
    human_decision = interrupt({
        "email_id": state.get("email_id", ""),
        "draft_response": state.get("draft_response", ""),   # 에스컬레이션이면 빈 문자열
        "urgency": cls.get("urgency"),
        "action": "이 답변을 검토하고 승인/수정해 주세요",
    })
    if human_decision.get("approved"):
        final = human_decision.get("edited_response", state.get("draft_response", ""))
        return Command(update={"draft_response": final, "log": log(state, "👤 human_review: 승인 → 발송")}, goto="send_reply")
    return Command(update={"log": log(state, "👤 human_review: 거절 → 수동 처리")}, goto=END)


def send_reply(state: EmailAgentState) -> dict:
    """[액션 스텝] 발송. 다룰 수 없는 에러는 그대로 띄워보낸다(전략 ④)."""
    try:
        return {"log": log(state, f"📤 send_reply: {state.get('sender_email', '')} 로 발송 완료")}
    except Exception:
        raise

print("노드 7개 정의 완료")

### 에러 전략 ① 재시도 · ② 되돌아오기 — 동작 확인

위 노드들에 ③·④ 가 들어 있으니, 나머지 두 전략을 작은 그래프로 직접 돌려 봅니다. **①** 은 `RetryPolicy` 가 일시적 실패를 자동 재시도하는 모습을, **②** 는 도구 실패 시 에러를 State 에 담아 `agent` 로 되돌아가 재계획하는 loop-back 패턴을 보여 줍니다.

In [ ]:
# ① 일시적 오류 — 두 번 실패 후 세 번째 성공 (RetryPolicy 가 실제로 재시도)
_attempts = []
def flaky_search(state: dict) -> dict:
    _attempts.append(1)
    n = len(_attempts)
    if n < 3:
        print(f"  🗄️ search 시도 {n}: 일시적 실패 → 재시도")
        raise ValueError("transient error")
    print(f"  🗄️ search 시도 {n}: 성공 ✅")
    return {"result": "ok"}

g1 = StateGraph(dict)
g1.add_node("search", flaky_search, retry_policy=RetryPolicy(max_attempts=3, initial_interval=0.01, retry_on=(ValueError,)))
g1.add_edge(START, "search"); g1.add_edge("search", END)
print("① 재시도 정책:"); g1.compile().invoke({})

# ② LLM 복구 가능 — 에러를 State 에 담아 agent 로 되돌아오기(loop-back)
class LoopState(TypedDict, total=False):
    tries: int; tool_error: str; done: bool

def agent(state: LoopState) -> Command[Literal["execute_tool", END]]:
    if state.get("done"):
        return Command(goto=END)
    if state.get("tool_error"):
        print(f"  🧠 agent: 직전 에러 '{state['tool_error']}' 보고 재계획")
    return Command(goto="execute_tool")

def execute_tool(state: LoopState) -> Command[Literal["agent"]]:
    tries = state.get("tries", 0) + 1
    if tries == 1:
        print("  ⚡ execute_tool: 실패 → 에러 저장 후 agent 로")
        return Command(update={"tries": tries, "tool_error": "bad argument"}, goto="agent")
    print("  ⚡ execute_tool: 성공 ✅")
    return Command(update={"tries": tries, "tool_error": "", "done": True}, goto="agent")

g2 = StateGraph(LoopState)
g2.add_node("agent", agent); g2.add_node("execute_tool", execute_tool)
g2.add_edge(START, "agent")
print("\n② 되돌아오기(loop-back):"); g2.compile().invoke({})
print()

## 4. 노드를 그래프로 연결 — 엣지는 최소, checkpointer 와 함께 (§4.2)

이제 7개 노드를 그래프로 잇습니다. 라우팅이 노드 안 `Command(goto)` 로 일어나므로 **고정 엣지는 단 3개**뿐입니다. `interrupt()` 로 사람 개입을 쓰려면 State 를 저장할 **checkpointer** 와 함께 컴파일해야 합니다(데모는 인메모리 `MemorySaver`).

In [ ]:
def build_graph(checkpointer=None):
    workflow = StateGraph(EmailAgentState)
    workflow.add_node("read_email", read_email)
    workflow.add_node("classify_intent", classify_intent)
    workflow.add_node("search_documentation", search_documentation,
                      retry_policy=RetryPolicy(max_attempts=3, initial_interval=1.0))  # 전략 ①
    workflow.add_node("bug_tracking", bug_tracking)
    workflow.add_node("draft_response", draft_response)
    workflow.add_node("human_review", human_review)
    workflow.add_node("send_reply", send_reply)
    # 고정 엣지는 셋뿐 — 나머지 라우팅은 노드 안 Command 가 한다
    workflow.add_edge(START, "read_email")
    workflow.add_edge("read_email", "classify_intent")
    workflow.add_edge("send_reply", END)
    return workflow.compile(checkpointer=checkpointer)

graph = build_graph()
print("그래프 컴파일 완료 · 노드:", list(graph.get_graph().nodes)[1:-1])

### 그래프 구조 시각화

`Command(goto)` 라우팅은 그래프에서 **점선**으로, `add_edge` 로 박은 고정 엣지는 **실선**으로 보입니다. 리포트 fig03 의 "실선=고정 엣지, 점선=노드 안 분기" 와 같은 그림입니다.

In [ ]:
from IPython.display import Image, display
try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    # 네트워크/graphviz 없으면 Mermaid 텍스트로 폴백
    print(graph.get_graph().draw_mermaid())

## 5. 멈췄다가 재개 — interrupt → checkpointer → resume (§4.3)

이제 완성된 그래프를 실제로 돌립니다. `interrupt()` 를 만나면 그래프는 모든 State 를 checkpointer 에 저장하고 멈춥니다. 같은 `thread_id` 로 `Command(resume=...)` 를 호출하면 **멈춘 지점부터** 이어집니다. 두 시나리오로 `human_review` 의 두 진입 경로(분류 직후 에스컬레이션 / 초안 후 검토)를 모두 봅니다.

In [ ]:
app = build_graph(checkpointer=MemorySaver())

def run_one(title, initial_state, human_resume):
    print("=" * 60); print(f"📨 {title}"); print("=" * 60)
    print(f"   원문: {initial_state['email_content']}")
    config = {"configurable": {"thread_id": initial_state["email_id"]}}
    result = app.invoke(initial_state, config)
    if "__interrupt__" in result:
        payload = result["__interrupt__"][0].value
        print(f"\n   ⏸️  human_review 일시정지 (urgency={payload.get('urgency')})")
        print(f"      초안: {payload.get('draft_response') or '(없음 — 에스컬레이션)'}")
        # 멈춘 사이 checkpointer 에 저장된 다음 실행 노드
        print(f"      checkpointer 저장 · 다음 노드: {app.get_state(config).next}")
        result = app.invoke(Command(resume=human_resume), config)
    print("\n   📜 실행 로그:")
    for line in result.get("log", []):
        print(f"      {line}")
    print()

# 시나리오 A: 긴급 청구 → 분류 직후 human_review 로 직행(초안 없음, 사람이 작성)
run_one("시나리오 A — 긴급 청구 (에스컬레이션)",
        {"email_content": "구독료가 두 번 청구됐어요! 급해요!", "sender_email": "customer@example.com", "email_id": "email_A01"},
        {"approved": True, "edited_response": "이중 청구 사과드립니다. 즉시 환불 처리했습니다."})

# 시나리오 B: 사용 문의 → 검색 → 초안 → human_review 검토 → 승인(초안 그대로)
run_one("시나리오 B — 사용 문의 (검색→초안→검토)",
        {"email_content": "비밀번호를 어떻게 변경하나요?", "sender_email": "user@example.com", "email_id": "email_B02"},
        {"approved": True})

## 6. 정리 — LangGraph 식 사고 6가지

이 노트북에서 만든 에이전트가 보여 준 원칙을 요약하면:

1. **개별 스텝으로 분해** — 노드 하나가 한 가지를 잘한다. 스트리밍·재개·디버깅이 여기서 나온다.
2. **State 는 공유 메모리** — 포맷된 문자열이 아니라 raw 데이터를 저장한다.
3. **노드는 함수** — State 를 받아 업데이트를 돌려준다. 라우팅이 필요하면 `Command(goto)` 로 다음 행선지까지 지정한다.
4. **에러는 흐름의 일부** — 재시도·되돌아오기·일시정지·띄워보내기로 나눈다.
5. **사람 입력은 1급** — `interrupt()` 가 무한정 멈추고 State 를 저장, 입력이 오면 그 자리에서 재개한다.
6. **그래프 구조는 자연히 생긴다** — 꼭 필요한 연결만 정의하고 라우팅은 노드가 한다.

> **노드를 얼마나 잘게?** 작게 쪼갤수록 실패 시 재실행 범위가 줄어든다(회복력). 노드가 많다고 느려지지 않는다 — 체크포인트는 기본 비동기로 쓰인다.

## 7. 로컬에서 띄워 Studio 로 보기 — `langgraph dev` (§5)

여기서 만든 그래프를 한 줄로 로컬 서버에 올려 Studio 로 디버깅할 수 있습니다. `scripts/` 폴더에 이미 `langgraph.json`(이 그래프를 `email_agent.py:graph` 로 등록)이 있으니, 터미널에서 다음을 실행하면 됩니다.

```bash
pip install -U "langgraph-cli[inmem]"
cd scripts
langgraph dev        # http://127.0.0.1:2024 · Studio URL 출력
```

Studio 에서 `human_review` 의 interrupt 를 눈으로 확인하고, 입력을 넣어 재개해 볼 수 있습니다. (`langgraph dev` 는 인메모리 개발 모드 — 운영은 영속 저장소를 갖춘 배포가 필요합니다.)

---

### 더 보기
- 완성 전체본 스크립트: `scripts/email_agent.py`
- 단계별 점진 빌드: `scripts/steps/01_decompose.py … 05_hitl.py`
- 공식 문서: [Thinking in LangGraph](https://docs.langchain.com/oss/python/langgraph/thinking-in-langgraph) · [Run a local server](https://docs.langchain.com/oss/python/langgraph/local-server) · [Changelog](https://docs.langchain.com/oss/python/releases/changelog)